# 03 — Demand Forecasting
**Project:** Supply Chain Intelligence System  
**Input:** `data/processed/master_orders.csv`  
**Goal:** Forecast 30/60/90 day demand by product category using Prophet and XGBoost. Compare models. Export forecast for Tableau.

---
## Approach
1. Build daily/weekly demand time series per category
2. Train Prophet model (handles seasonality natively)
3. Train XGBoost model (feature-based)
4. Compare MAPE — pick best model
5. Export 90-day forecast to CSV

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

master = pd.read_csv('../data/processed/master_orders.csv', parse_dates=['order_purchase_timestamp'])
print(f'Loaded {len(master):,} rows')

## 1. Build Weekly Demand Time Series

In [ ]:
# Aggregate to weekly demand (order count) overall
master['week'] = master['order_purchase_timestamp'].dt.to_period('W').dt.start_time

weekly_demand = master.groupby('week').agg(
    order_count = ('order_id', 'nunique'),
    revenue     = ('payment_value', 'sum')
).reset_index()

# Remove first and last partial weeks
weekly_demand = weekly_demand.iloc[1:-1]

print(f'Weekly demand series: {len(weekly_demand)} weeks')
print(f'Date range: {weekly_demand["week"].min()} to {weekly_demand["week"].max()}')
print(f'Avg orders/week: {weekly_demand["order_count"].mean():.0f}')

weekly_demand.tail()

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(weekly_demand['week'], weekly_demand['order_count'], color='steelblue', linewidth=1.5)
plt.title('Weekly Order Volume — Olist Brazil', fontweight='bold')
plt.xlabel('Week')
plt.ylabel('Orders')
plt.tight_layout()
plt.savefig('../reports/demand_timeseries.png', dpi=150)
plt.show()

## 2. Prophet Model

In [ ]:
# Prophet requires columns named 'ds' and 'y'
prophet_df = weekly_demand[['week', 'order_count']].rename(
    columns={'week': 'ds', 'order_count': 'y'}
)

# Train/test split — last 8 weeks as holdout
HOLDOUT_WEEKS = 8
train_prophet = prophet_df.iloc[:-HOLDOUT_WEEKS]
test_prophet  = prophet_df.iloc[-HOLDOUT_WEEKS:]

# Fit model
model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    changepoint_prior_scale=0.05  # Controls trend flexibility
)
model_prophet.fit(train_prophet)

# Predict on test period
future_test = model_prophet.make_future_dataframe(periods=HOLDOUT_WEEKS, freq='W')
forecast_test = model_prophet.predict(future_test)

# Evaluate
y_true = test_prophet['y'].values
y_pred = forecast_test.tail(HOLDOUT_WEEKS)['yhat'].values
y_pred = np.maximum(y_pred, 0)  # No negative orders

mape_prophet = mean_absolute_percentage_error(y_true, y_pred)
print(f'Prophet MAPE: {mape_prophet*100:.1f}%')

## 3. XGBoost Model

In [ ]:
# Feature engineering for XGBoost
xgb_df = weekly_demand.copy()
xgb_df['week_num']   = xgb_df['week'].dt.isocalendar().week.astype(int)
xgb_df['month']      = xgb_df['week'].dt.month
xgb_df['quarter']    = xgb_df['week'].dt.quarter
xgb_df['lag_1']      = xgb_df['order_count'].shift(1)
xgb_df['lag_2']      = xgb_df['order_count'].shift(2)
xgb_df['lag_4']      = xgb_df['order_count'].shift(4)
xgb_df['rolling_4w'] = xgb_df['order_count'].shift(1).rolling(4).mean()
xgb_df = xgb_df.dropna()

FEATURES = ['week_num', 'month', 'quarter', 'lag_1', 'lag_2', 'lag_4', 'rolling_4w']
TARGET   = 'order_count'

X = xgb_df[FEATURES]
y = xgb_df[TARGET]

X_train, X_test = X.iloc[:-HOLDOUT_WEEKS], X.iloc[-HOLDOUT_WEEKS:]
y_train, y_test = y.iloc[:-HOLDOUT_WEEKS], y.iloc[-HOLDOUT_WEEKS:]

model_xgb = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)
mape_xgb = mean_absolute_percentage_error(y_test, y_pred_xgb)
print(f'XGBoost MAPE: {mape_xgb*100:.1f}%')

## 4. Model Comparison

In [ ]:
print('=== MODEL COMPARISON ===')
print(f'Prophet MAPE: {mape_prophet*100:.1f}%')
print(f'XGBoost MAPE: {mape_xgb*100:.1f}%')
winner = 'Prophet' if mape_prophet < mape_xgb else 'XGBoost'
print(f'Winner: {winner}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

test_dates = weekly_demand['week'].iloc[-HOLDOUT_WEEKS:]
for ax, (name, pred) in zip(axes, [('Prophet', y_pred), ('XGBoost', y_pred_xgb)]):
    ax.plot(test_dates.values, y_true, label='Actual', color='black', linewidth=2)
    ax.plot(test_dates.values, pred, label=f'{name} forecast', linestyle='--', linewidth=2)
    ax.set_title(f'{name} — MAPE: {mean_absolute_percentage_error(y_true, pred)*100:.1f}%')
    ax.legend()
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/model_comparison.png', dpi=150)
plt.show()

## 5. Generate 90-Day Forecast

In [ ]:
# Retrain Prophet on full data, forecast 13 weeks (90 days)
model_final = Prophet(yearly_seasonality=True, weekly_seasonality=True, changepoint_prior_scale=0.05)
model_final.fit(prophet_df)

future = model_final.make_future_dataframe(periods=13, freq='W')
forecast = model_final.predict(future)

# Extract forecast period only
forecast_out = forecast[forecast['ds'] > prophet_df['ds'].max()][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
forecast_out.columns = ['week', 'forecast_orders', 'forecast_lower', 'forecast_upper']
forecast_out['forecast_orders'] = np.maximum(forecast_out['forecast_orders'], 0).round(0).astype(int)
forecast_out['forecast_lower']  = np.maximum(forecast_out['forecast_lower'],  0).round(0).astype(int)
forecast_out['forecast_upper']  = forecast_out['forecast_upper'].round(0).astype(int)

forecast_out.to_csv('../data/processed/demand_forecast_90day.csv', index=False)
print('Exported: data/processed/demand_forecast_90day.csv')
print(forecast_out.to_string(index=False))